# Predict early-season FPL performance from the previous season

Build one row per player from 2024/25 performance, target mean `total_points` over gameweeks 1–5 of 2025/26, compare the four MVP approaches, then apply the selected model to 2025/26 aggregates for the next draft.

> A player-level holdout is used because the target is now one value per player rather than one value per player-gameweek.


In [1]:
import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

from fantasy_football.fpl_api.get_live_data import LivePlayerData
from fantasy_football.fpl_api.get_performance_data import get_most_recent_gw_points
from fantasy_football.model.training.utils import load_gw_data
from fantasy_football.utils import create_optimal_team

RANDOM_STATE = 42
EARLY_GAMEWEEKS = 10
DECAY = 0.90


## Load feature and target seasons


In [2]:
season_2024_25 = load_gw_data("2024-25")
season_2025_26 = load_gw_data("2025-26")

print("2024/25:", season_2024_25.shape)
print("2025/26:", season_2025_26.shape)


2024/25: (27605, 49)
2025/26: (29757, 46)


## Feature engineering

For each MVP numeric metric, calculate its season sum, appearance mean, appearance standard deviation, and exponentially decayed appearance mean. Later gameweeks receive more weight.


In [3]:
mvp_numeric_features = [
    "assists", "clean_sheets", "creativity", "goals_conceded", "goals_scored",
    "ict_index", "influence", "own_goals", "penalties_missed", "penalties_saved",
    "red_cards", "saves", "selected", "starts", "threat", "transfers_balance",
    "value", "yellow_cards", "clearances_blocks_interceptions",
    "defensive_contribution", "recoveries", "tackles",
]

def latest_non_null(series):
    values = series.dropna()
    return values.iloc[-1] if len(values) else np.nan

def decayed_mean(values, decay=DECAY):
    values = pd.to_numeric(values, errors="coerce").dropna().to_numpy(dtype=float)
    if not len(values):
        return np.nan
    weights = decay ** np.arange(len(values) - 1, -1, -1)
    return np.average(values, weights=weights)

def build_season_features(gameweeks, feature_columns=mvp_numeric_features):
    data = gameweeks.sort_values(["name", "GW"]).copy()
    available = [column for column in feature_columns if column in data.columns]
    rows = []
    for name, player_rows in data.groupby("name", sort=False):
        appearances = player_rows.loc[player_rows["minutes"].fillna(0).gt(0)]
        row = {
            "name": name,
            "position": (
                latest_non_null(player_rows["position"])
                if "position" in player_rows else "__MISSING__"
            ),
            "team": (
                latest_non_null(player_rows["team"])
                if "team" in player_rows else "__MISSING__"
            ),
            "games_available": player_rows["GW"].nunique(),
            "appearances": appearances["GW"].nunique(),
            "minutes_sum": player_rows["minutes"].sum(),
        }
        for column in available:
            all_values = pd.to_numeric(player_rows[column], errors="coerce")
            played_values = pd.to_numeric(appearances[column], errors="coerce")
            row[f"{column}_sum"] = all_values.sum(min_count=1)
            row[f"{column}_mean"] = played_values.mean()
            row[f"{column}_std"] = played_values.std(ddof=0)
            row[f"{column}_decayed_mean"] = decayed_mean(played_values)
        rows.append(row)
    return pd.DataFrame(rows)

features_2024_25 = build_season_features(season_2024_25)
features_2024_25.head()


,name,position,team,games_available,appearances,minutes_sum,assists_sum,assists_mean,assists_std,assists_decayed_mean,...,transfers_balance_std,transfers_balance_decayed_mean,value_sum,value_mean,value_std,value_decayed_mean,yellow_cards_sum,yellow_cards_mean,yellow_cards_std,yellow_cards_decayed_mean
0,Aaron Anselmino,DEF,Chelsea,14,0,0,0,NaN,NaN,NaN,...,NaN,NaN,560,NaN,NaN,NaN,0,NaN,NaN,NaN
1,Aaron Cresswell,DEF,West Ham,38,18,818,0,0.000000,0.000000,0.000000,...,7773.953739,1097.538000,1497,39.277778,0.447903,39.122475,3,0.166667,0.372678,0.167806
2,Aaron Hickey,DEF,Brentford,38,0,0,0,NaN,NaN,NaN,...,NaN,NaN,1647,NaN,NaN,NaN,0,NaN,NaN,NaN
3,Aaron Ramsdale,GK,Southampton,38,30,2700,0,0.000000,0.000000,0.000000,...,12768.076043,1242.836846,1674,44.000000,0.000000,44.000000,2,0.066667,0.249444,0.040919
4,Aaron Wan-Bissaka,DEF,West Ham,38,36,3154,6,0.166667,0.372678,0.322007,...,67654.362825,17252.075486,1705,44.861111,0.713083,44.690646,1,0.027778,0.164336,0.018957


## Target: mean points in gameweeks 1–5 of 2025/26


In [4]:
early_2025_26 = season_2025_26.loc[
    season_2025_26["GW"].between(1, EARLY_GAMEWEEKS)
].copy()
player_gw_points = (
    early_2025_26.groupby(["name", "GW"], as_index=False)["total_points"].sum()
)
target_2025_26 = (
    player_gw_points.groupby("name", as_index=False)
    .agg(
        target_avg_points_first_5_gw=("total_points", "mean"),
        target_total_points_first_5_gw=("total_points", "sum"),
        target_gw_rows=("GW", "count"),
    )
)

model_data = features_2024_25.merge(target_2025_26, on="name", how="inner")
print(f"Matched {len(model_data):,} players across seasons")
model_data[["name", "target_avg_points_first_5_gw"]].sort_values(
    "target_avg_points_first_5_gw", ascending=False
).head(10)


Matched 466 players across seasons


,name,target_avg_points_first_5_gw
125,Erling Haaland,9.8
144,Gabriel dos Santos Magalhães,8.0
29,Antoine Semenyo,7.5
236,Jurriën Timber,6.6
293,Marc Guéhi,6.4
95,Declan Rice,6.3
379,Riccardo Calafiori,6.0
330,Moisés Caicedo Corozo,5.8
322,Micky van de Ven,5.7
54,Bryan Mbeumo,5.3


## Player-level train/validation split


In [5]:
target_column = "target_avg_points_first_5_gw"
categorical_columns = ["position", "team"]
excluded_columns = {
    "name", target_column, "target_total_points_first_5_gw", "target_gw_rows"
}
model_features = [column for column in model_data.columns if column not in excluded_columns]

train_index, valid_index = train_test_split(
    model_data.index, test_size=0.20, random_state=RANDOM_STATE
)
X_train = model_data.loc[train_index, model_features].copy()
X_valid = model_data.loc[valid_index, model_features].copy()
y_train = model_data.loc[train_index, target_column].copy()
y_valid = model_data.loc[valid_index, target_column].copy()
for frame in (X_train, X_valid):
    frame[categorical_columns] = frame[categorical_columns].fillna("__MISSING__").astype(str)

print(f"Train players: {len(X_train):,}; validation players: {len(X_valid):,}")


Train players: 372; validation players: 94


## Four modelling approaches from the MVP


In [6]:
models = {}
validation_predictions = {}

dummy_model = DummyRegressor(strategy="mean").fit(X_train, y_train)
models["Dummy"] = dummy_model
validation_predictions["Dummy"] = dummy_model.predict(X_valid)

def fit_catboost(X, y, sample_weight=None):
    model = CatBoostRegressor(
        iterations=163, learning_rate=0.03, depth=6, loss_function="RMSE",
        random_seed=RANDOM_STATE, verbose=False, allow_writing_files=False,
    )
    model.fit(X, y, cat_features=categorical_columns, sample_weight=sample_weight)
    return model

unweighted_model = fit_catboost(X_train, y_train)
models["Unweighted"] = unweighted_model
validation_predictions["Unweighted"] = unweighted_model.predict(X_valid)

basic_weights = pd.Series(1.0, index=y_train.index)
basic_weights.loc[y_train > 6] = 4.0
basic_model = fit_catboost(X_train, y_train, basic_weights)
models["BASIC weighting"] = basic_model
validation_predictions["BASIC weighting"] = basic_model.predict(X_valid)

target_percentiles = y_train.rank(pct=True, method="average")
percentile_weights = pd.cut(
    target_percentiles, bins=[0, .25, .50, .75, .90, 1.0],
    labels=[1.0, 1.5, 2.0, 3.0, 4.0], include_lowest=True,
).astype(float)
percentile_model = fit_catboost(X_train, y_train, percentile_weights)
models["Percentile weighting"] = percentile_model
validation_predictions["Percentile weighting"] = percentile_model.predict(X_valid)


## Comparison table


In [7]:
def evaluate_predictions(predictions, top_n=20):
    evaluation = pd.DataFrame({"actual": y_valid, "predicted": predictions})
    n = min(top_n, len(evaluation))
    predicted_top = evaluation.nlargest(n, "predicted")
    actual_top_indices = set(evaluation.nlargest(n, "actual").index)
    return {
        "MAE": mean_absolute_error(evaluation["actual"], evaluation["predicted"]),
        "RMSE": root_mean_squared_error(evaluation["actual"], evaluation["predicted"]),
        "R2": r2_score(evaluation["actual"], evaluation["predicted"]),
        "top_20_avg_actual_points": predicted_top["actual"].mean(),
        "top_20_hit_rate": predicted_top.index.isin(actual_top_indices).mean(),
        "top_20_oracle_regret": (
            evaluation.nlargest(n, "actual")["actual"].mean()
            - predicted_top["actual"].mean()
        ),
    }

comparison_table = (
    pd.DataFrame.from_dict(
        {name: evaluate_predictions(preds) for name, preds in validation_predictions.items()},
        orient="index",
    )
    .rename_axis("model").reset_index()
    .sort_values(["top_20_avg_actual_points", "MAE"], ascending=[False, True])
    .reset_index(drop=True)
)
selected_model_name = comparison_table.loc[0, "model"]
comparison_table.round(3)


,model,MAE,RMSE,R2,top_20_avg_actual_points,top_20_hit_rate,top_20_oracle_regret
0,Unweighted,1.018,1.349,0.316,2.565,0.45,1.430
1,BASIC weighting,1.059,1.429,0.233,2.540,0.45,1.455
2,Percentile weighting,1.128,1.434,0.227,2.490,0.45,1.505
3,Dummy,1.435,1.632,-0.002,1.080,0.20,2.915


## Next-season draft using 2025/26 performance

Refit the selected approach on every matched player, build the same features from 2025/26, and predict each returning player's early-season average. Live FPL data supplies current eligibility, club, and price before squad optimisation.


In [8]:
X_all = model_data[model_features].copy()
y_all = model_data[target_column].copy()
X_all[categorical_columns] = X_all[categorical_columns].fillna("__MISSING__").astype(str)

if selected_model_name == "Dummy":
    final_model = DummyRegressor(strategy="mean").fit(X_all, y_all)
else:
    final_weights = None
    if selected_model_name == "BASIC weighting":
        final_weights = pd.Series(1.0, index=y_all.index)
        final_weights.loc[y_all > 6] = 4.0
    elif selected_model_name == "Percentile weighting":
        percentiles = y_all.rank(pct=True, method="average")
        final_weights = pd.cut(
            percentiles, bins=[0, .25, .50, .75, .90, 1.0],
            labels=[1.0, 1.5, 2.0, 3.0, 4.0], include_lowest=True,
        ).astype(float)
    final_model = fit_catboost(X_all, y_all, final_weights)

features_2025_26 = build_season_features(season_2025_26)
forecast_matrix = features_2025_26.reindex(columns=model_features).copy()
forecast_matrix[categorical_columns] = (
    forecast_matrix[categorical_columns].fillna("__MISSING__").astype(str)
)
features_2025_26["prediction"] = final_model.predict(forecast_matrix)
features_2025_26[["name", "prediction"]].sort_values(
    "prediction", ascending=False
).head(20)


,name,prediction
223,Erling Haaland,4.061824
530,Marcus Tavernier,3.717364
524,Marc Guéhi,3.620444
177,Declan Rice,3.444186
260,Gabriel dos Santos Magalhães,3.439426
349,Jan Paul van Hecke,3.435079
90,Bernardo Mota Veiga de Carvalho e Silva,3.427543
590,Morgan Gibbs-White,3.422429
424,Jurriën Timber,3.420831
548,Matheus Nunes,3.409026


In [13]:
live_data = LivePlayerData()
forecast_data = features_2025_26[["name", "prediction"]].copy()
forecast_data["player_id"] = forecast_data["name"].map(live_data.get_player_id)
forecast_data["web_name"] = forecast_data["name"].map(
    live_data.get_player_web_name
)
forecast_data["position"] = forecast_data["name"].map(live_data.get_live_player_position)
forecast_data["team"] = forecast_data["name"].map(live_data.get_live_player_team)
forecast_data["value"] = forecast_data["name"].map(live_data.get_live_player_cost)
forecast_data = forecast_data.dropna(
    subset=["player_id", "web_name", "position", "team", "value", "prediction"]
).sort_values("prediction", ascending=False)



excluded_player_names = ["Jurriën Timber", "Morgan Gibbs-White", "Junior Kroupi"]



optimal_team = create_optimal_team(
    forecast_data, "prediction", excluded_player_names=excluded_player_names
)
optimal_team["last_gw_points"] = optimal_team["player_id"].map(
    get_most_recent_gw_points
)
optimal_team.sort_values(["position", "prediction"], ascending=[True, False])


,name,prediction,player_id,web_name,position,team,value,last_gw_points
2,Marc Guéhi,3.620444,388,Guéhi,DEF,Man City,60.0,10
4,Gabriel dos Santos Magalhães,3.439426,4,Gabriel,DEF,Arsenal,80.0,5
5,Jan Paul van Hecke,3.435079,112,Van Hecke,DEF,Spurs,50.0,1
8,Marcos Senesi Barón,3.219827,498,Senesi,DEF,Spurs,60.0,3
9,Nathan Collins,3.183298,84,Collins,DEF,Brentford,55.0,6
0,Erling Haaland,4.061824,411,Haaland,FWD,Man City,155.0,2
11,Dominic Calvert-Lewin,2.974420,346,Calvert-Lewin,FWD,Leeds,60.0,1
12,Danny Welbeck,2.870777,136,Welbeck,FWD,Chelsea,60.0,0
13,Bart Verbruggen,2.679449,109,Verbruggen,GKP,Brighton,45.0,6
14,Đorđe Petrović,2.545528,57,Petrović,GKP,Bournemouth,45.0,2


In [12]:
pd.Series({
    "players": len(optimal_team),
    "cost_millions": optimal_team["value"].sum() / 10,
    "predicted_avg_points": optimal_team["prediction"].sum(),
    "week_1_score" : optimal_team['last_gw_points'].sum(),
    "selected_model": selected_model_name,
})


players                         15
cost_millions                 99.9
predicted_avg_points     48.972876
week_1_score                    48
selected_model          Unweighted
dtype: object